# AI Agent Skills 各领域推荐集合构建与可视化分析

## 项目背景

随着 AI Agent 应用场景不断扩展，Skill 可以理解为面向某一类具体任务的能力模块，例如代码开发、营销增长、科研分析、文档处理、视频理解、RAG 检索、安全审计等。不同领域的 Skill 数量多、来源杂、质量差异明显，如果直接查看原始列表，很难快速判断每个领域有哪些值得优先使用的 Skill。

因此，本项目的核心目标是基于已完成的数据预处理结果，构建一个“各个领域的 Skill 推荐集合”。Notebook 会直接读取数据文件，在代码内部完成统计表生成和可视化绘图，不依赖外部图表目录。


## 一、研究目标与任务定义

本项目的主要研究目标是：**构建一个面向不同应用领域的 AI Agent Skills 推荐集合**。

具体任务包括：

1. 对原始 Skill 数据进行整理，区分官方 Skill 与社区 Skill。
2. 针对社区 Skill，根据名称、描述和类别信息划分应用领域。
3. 建立综合评分规则，从多个维度评价每个 Skill 的推荐价值。
4. 在每个领域中筛选出推荐 Skill，形成“领域—推荐 Skill—具体作用—推荐等级”的集合。
5. 通过图表和表格展示推荐集合的构建结果。


## 二、运行环境与数据文件检查

本 Notebook 不再依赖 generate_paper_charts.py、outputs/figures、outputs/tables 或 requirements.txt。运行时只需要 Notebook 本身和 data 目录中的核心 JSON 数据。

本 Notebook 使用 Python 运行，主要依赖以下常见数据分析库：

- pandas
- matplotlib
- seaborn
- IPython

这些库通常在 Anaconda 或 Jupyter 环境中已经包含。若本地环境缺少，可在命令行中安装：

    pip install pandas matplotlib seaborn ipython


In [ ]:
import json
import math
import sys
from pathlib import Path
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_PATH = DATA_DIR / 'visualization_data.json'
SKILLS_PATH = DATA_DIR / 'skills.json'
METRICS_PATH = DATA_DIR / 'github_repo_metrics.json'

required_paths = [DATA_PATH, SKILLS_PATH, METRICS_PATH]
check_rows = []
for path in required_paths:
    check_rows.append({
        '路径': str(path.relative_to(PROJECT_ROOT)),
        '是否存在': path.exists(),
        '大小KB': round(path.stat().st_size / 1024, 2) if path.exists() else None,
    })
check_df = pd.DataFrame(check_rows)
display(check_df)

missing = [row['路径'] for row in check_rows if not row['是否存在']]
if missing:
    raise FileNotFoundError('缺少必要数据文件：' + ', '.join(missing))

with DATA_PATH.open('r', encoding='utf-8') as f:
    data = json.load(f)
with SKILLS_PATH.open('r', encoding='utf-8') as f:
    raw_skills_data = json.load(f)
with METRICS_PATH.open('r', encoding='utf-8') as f:
    github_metrics_data = json.load(f)

print('Python 版本：', sys.version.split()[0])
print('当前工作目录：', PROJECT_ROOT)
print('运行时间：', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


## 三、数据文件与字段说明

本项目使用的数据来自 GitHub 公开项目 VoltAgent/awesome-agent-skills，并结合 GitHub 仓库补充指标进行扩展。主要数据文件如下：

| 文件 | 作用 |
|---|---|
| data/visualization_data.json | 核心分析数据，包含总览、评分模型、官方机构、社区分类、推荐结果和社区 Skill 明细 |
| data/skills.json | 原始/清洗后的 Skill 主数据，包含名称、描述、链接、类别、来源等字段 |
| data/github_repo_metrics.json | GitHub 仓库指标数据，包含 Stars、Forks、语言、更新时间、README 长度等 |


In [ ]:
overview = data['overview']
skills_df = pd.DataFrame(raw_skills_data.get('skills', []))
community_raw_df = pd.DataFrame(data.get('enriched_community_skills', []))
official_company_df = pd.DataFrame(data.get('official_companies', []))
category_df = pd.DataFrame(data.get('community_categories', []))

display(pd.DataFrame([overview]))
print('Skill 主数据维度：', skills_df.shape)
print('社区 Skill 明细维度：', community_raw_df.shape)
print('官方机构统计维度：', official_company_df.shape)
print('社区分类统计维度：', category_df.shape)
display(skills_df.head(5))
display(community_raw_df[['name', 'scenario_category', 'recommend_score', 'rating_name', 'effect', 'link']].head(10))


## 四、数据整理函数

下面的代码把 JSON 中嵌套的 GitHub 指标和评分明细展开为表格字段，后续图表都基于这些 DataFrame 直接生成。


In [ ]:
def setup_style():
    sns.set_theme(style='whitegrid')
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Arial Unicode MS', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.dpi'] = 130
    plt.rcParams['axes.titlesize'] = 15
    plt.rcParams['axes.labelsize'] = 11

def pct(value, total):
    return round(value / total * 100, 2) if total else 0

def parse_date(value):
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace('Z', '+00:00'))
    except ValueError:
        return None

def get_community_df(data):
    rows = []
    for item in data.get('enriched_community_skills', []):
        metrics = item.get('github_metrics') or {}
        detail = item.get('score_detail') or {}
        updated = parse_date(metrics.get('pushed_at') or metrics.get('updated_at'))
        days_since_update = None
        if updated:
            days_since_update = (datetime.now(timezone.utc) - updated.astimezone(timezone.utc)).days
        rows.append({
            'name': item.get('name'),
            'link': item.get('link'),
            'scenario_category': item.get('scenario_category'),
            'recommend_score': item.get('recommend_score') or 0,
            'rating_name': item.get('rating_name'),
            'rating_stars': item.get('rating_stars'),
            'effect': item.get('effect'),
            'language': metrics.get('language') or '未知',
            'stars': metrics.get('stars') or 0,
            'forks': metrics.get('forks') or 0,
            'readme_length': metrics.get('readme_length') or 0,
            'days_since_update': days_since_update,
            '场景匹配度': detail.get('scenario_match') or 0,
            'GitHub热度': detail.get('github_popularity') or 0,
            '维护活跃度': detail.get('maintenance') or 0,
            '文档质量': detail.get('documentation') or 0,
            '项目可信度': detail.get('trust') or 0,
            '功能明确度': detail.get('clarity') or 0,
        })
    return pd.DataFrame(rows)

setup_style()
community_df = get_community_df(data)
display(community_df.head(10))


## 五、数据质量检查与基础统计


In [ ]:
missing_summary = pd.DataFrame({
    '字段': skills_df.columns,
    '缺失数量': [skills_df[col].isna().sum() for col in skills_df.columns],
    '缺失比例': [round(skills_df[col].isna().mean(), 4) for col in skills_df.columns]
}).sort_values('缺失数量', ascending=False)
display(missing_summary)

source_count = skills_df['source'].value_counts().reset_index()
source_count.columns = ['来源类型', '数量']
display(source_count)

category_overview = category_df[['category', 'count', 'recommended_count']].sort_values('count', ascending=False)
display(category_overview)


## 六、推荐评分模型说明


In [ ]:
score_model = data.get('score_model', {})
dimensions = score_model.get('dimensions', {})
if isinstance(dimensions, dict):
    score_rows = []
    for key, value in dimensions.items():
        row = {'维度': key}
        if isinstance(value, dict):
            row.update(value)
        else:
            row['说明/权重'] = value
        score_rows.append(row)
    score_model_df = pd.DataFrame(score_rows)
else:
    score_model_df = pd.DataFrame(dimensions)

print('评分模型总分：', score_model.get('total', 100))
display(score_model_df)
display(community_df[['recommend_score']].describe())


## 七、社区 Skill 分领域推荐集合生成


In [ ]:
rows = []
for category in data.get('community_categories', []):
    category_name = category.get('category')
    for skill in category.get('recommended', []):
        rows.append({
            '场景类别': category_name,
            'Skill名称': skill.get('name'),
            '推荐得分': skill.get('recommend_score'),
            '推荐等级': skill.get('rating_name'),
            '具体作用': skill.get('effect'),
            '链接': skill.get('link')
        })
recommendation_df = pd.DataFrame(rows)
display(recommendation_df.head(30))

recommendation_summary = (
    recommendation_df.groupby('场景类别')
    .agg(推荐数量=('Skill名称', 'count'), 平均得分=('推荐得分', 'mean'), 最高得分=('推荐得分', 'max'))
    .reset_index()
    .sort_values(['推荐数量', '最高得分'], ascending=[False, False])
)
recommendation_summary['平均得分'] = recommendation_summary['平均得分'].round(2)
display(recommendation_summary)


## 八、可视化分析一：数据来源与官方供给


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.2))
values = [overview['official_total'], overview['community_total']]
labels = ['官方 Skill', '社区 Skill']
ax.pie(values, labels=labels, autopct='%1.1f%%', startangle=90,
       colors=['#2563eb', '#7c3aed'], wedgeprops={'linewidth': 1, 'edgecolor': 'white', 'width': 0.45},
       textprops={'fontsize': 11})
ax.text(0, 0, f"N={overview['total']}", ha='center', va='center', fontsize=14, weight='bold')
ax.set_title('Skill 数据来源结构')
plt.show()
display(Markdown('**图1 Skill 数据来源结构。** 该图展示官方 Skill 与社区 Skill 在总体数据中的占比。'))


In [ ]:
df = official_company_df.head(15)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=df, y='company', x='count', ax=ax, color='#2563eb')
ax.set_title('官方机构 Skill 供给数量 Top 15')
ax.set_xlabel('Skill 数量')
ax.set_ylabel('官方机构')
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=9)
plt.show()
display(Markdown('**图2 官方机构 Skill 供给数量 Top 15。** 该图用于分析官方 Skill 生态中的头部机构集中情况。'))


## 九、可视化分析二：场景分布与评分质量


In [ ]:
scenario_plot_df = category_df[category_df['count'] > 0][['category', 'count']].sort_values('count', ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=scenario_plot_df, y='category', x='count', ax=ax, hue='category', palette='viridis', legend=False)
ax.set_title('社区 Skill 应用场景数量分布 Top 15')
ax.set_xlabel('Skill 数量')
ax.set_ylabel('应用场景')
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=9)
plt.show()
display(Markdown('**图3 社区 Skill 应用场景数量分布 Top 15。** 该图展示社区 Skill 的主要应用方向。'))

scenario_table = category_df[['category', 'count', 'recommended_count']].rename(columns={'category': '场景类别', 'count': 'Skill数量', 'recommended_count': '推荐数量'}).sort_values('Skill数量', ascending=False)
display(scenario_table)


In [ ]:
order = ['夯', '顶级', '人上人', 'NPC', '拉完了']
counts = community_df['rating_name'].value_counts().reindex(order).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(8, 5.2))
bars = ax.bar(counts.index, counts.values, color=['#16a34a', '#2563eb', '#7c3aed', '#f97316', '#ef4444'])
ax.set_title('社区 Skill 推荐评级分布')
ax.set_xlabel('推荐评级')
ax.set_ylabel('Skill 数量')
ax.bar_label(bars, labels=[f'{v}\n({pct(v, len(community_df))}%)' for v in counts.values], padding=3, fontsize=9)
plt.show()
display(Markdown('**图4 社区 Skill 推荐评级分布。** 该图用于说明社区项目整体推荐质量的等级结构。'))


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.4))
sns.histplot(community_df['recommend_score'], bins=18, kde=True, ax=ax, color='#2563eb')
ax.axvline(community_df['recommend_score'].mean(), color='#ef4444', linestyle='--', label=f"均值={community_df['recommend_score'].mean():.1f}")
ax.set_title('社区 Skill 综合推荐得分分布')
ax.set_xlabel('综合推荐得分')
ax.set_ylabel('频数')
ax.legend()
plt.show()
display(Markdown('**图5 社区 Skill 综合推荐得分分布。** 该图用于观察推荐分数的集中程度与离散情况。'))


In [ ]:
top_categories = community_df['scenario_category'].value_counts().head(12).index
plot_df = community_df[community_df['scenario_category'].isin(top_categories)].copy()
order = plot_df.groupby('scenario_category')['recommend_score'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(11, 7))
sns.boxplot(data=plot_df, y='scenario_category', x='recommend_score', order=order, ax=ax, color='#bfdbfe')
sns.stripplot(data=plot_df, y='scenario_category', x='recommend_score', order=order, ax=ax, size=3, color='#2563eb', alpha=0.45)
ax.set_title('主要应用场景下推荐得分差异')
ax.set_xlabel('综合推荐得分')
ax.set_ylabel('应用场景')
plt.show()
display(Markdown('**图6 主要应用场景下推荐得分差异。** 该图说明推荐时应坚持同场景比较。'))


## 十、可视化分析三：技术栈、Stars、维护和文档质量


In [ ]:
lang = community_df['language'].value_counts().head(12).reset_index()
lang.columns = ['语言', '数量']
fig, ax = plt.subplots(figsize=(8.5, 5.5))
sns.barplot(data=lang, x='数量', y='语言', ax=ax, hue='语言', palette='mako', legend=False)
ax.set_title('社区 GitHub 项目主要开发语言分布 Top 12')
ax.set_xlabel('项目数量')
ax.set_ylabel('开发语言')
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=9)
plt.show()
display(Markdown('**图7 社区 GitHub 项目主要开发语言分布 Top 12。** 该图用于说明社区 Skill 的技术栈分布。'))


In [ ]:
plot_df = community_df.copy()
plot_df['log_stars'] = plot_df['stars'].apply(lambda x: math.log10(x + 1))
fig, ax = plt.subplots(figsize=(8.5, 5.8))
sns.scatterplot(data=plot_df, x='log_stars', y='recommend_score', hue='rating_name', size='forks', sizes=(20, 220), alpha=0.72, ax=ax)
ax.set_title('GitHub Star 热度与推荐得分关系')
ax.set_xlabel('log10(stars + 1)')
ax.set_ylabel('综合推荐得分')
ax.legend(title='评级/分叉数', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.show()
display(Markdown('**图8 GitHub Star 热度与推荐得分关系。** Stars 能反映热度，但不能完全代表推荐价值。'))


In [ ]:
dimensions = ['场景匹配度', 'GitHub热度', '维护活跃度', '文档质量', '项目可信度', '功能明确度']
top_categories = community_df['scenario_category'].value_counts().head(12).index
heat = community_df[community_df['scenario_category'].isin(top_categories)].groupby('scenario_category')[dimensions].mean()
heat = heat.loc[heat.mean(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(9.5, 7))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlGnBu', linewidths=0.5, ax=ax)
ax.set_title('不同应用场景的评分维度均值热力图')
ax.set_xlabel('评分维度')
ax.set_ylabel('应用场景')
plt.show()
display(Markdown('**图9 不同应用场景的评分维度均值热力图。** 该图用于解释不同场景在热度、维护、文档等维度上的差异。'))


In [ ]:
bins = [-1, 30, 90, 180, 365, 10000]
labels = ['30天内', '31-90天', '91-180天', '181-365天', '365天以上']
valid = community_df.dropna(subset=['days_since_update']).copy()
valid['维护间隔'] = pd.cut(valid['days_since_update'], bins=bins, labels=labels)
counts = valid['维护间隔'].value_counts().reindex(labels).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(8.5, 5.2))
bars = ax.bar(counts.index.astype(str), counts.values, color='#0891b2')
ax.set_title('社区 GitHub 项目最近维护时间分布')
ax.set_xlabel('距最近更新/推送时间')
ax.set_ylabel('项目数量')
ax.bar_label(bars, padding=3, fontsize=9)
plt.show()
display(Markdown('**图10 社区 GitHub 项目最近维护时间分布。** 该图用于衡量社区 Skill 项目的维护活跃程度。'))


In [ ]:
plot_df = community_df.copy()
plot_df['log_readme_length'] = plot_df['readme_length'].apply(lambda x: math.log10(x + 1))
fig, ax = plt.subplots(figsize=(8.5, 5.6))
sns.regplot(data=plot_df, x='log_readme_length', y='recommend_score', ax=ax, scatter_kws={'alpha': 0.45, 's': 32}, line_kws={'color': '#ef4444'})
ax.set_title('README 文档长度与推荐得分关系')
ax.set_xlabel('log10(README 长度 + 1)')
ax.set_ylabel('综合推荐得分')
plt.show()
display(Markdown('**图11 README 文档长度与推荐得分关系。** 该图用于讨论文档质量与项目推荐结果之间的关系。'))


## 十一、推荐结果表


In [ ]:
top_table = community_df.sort_values('recommend_score', ascending=False).head(30)[[
    'name', 'scenario_category', 'recommend_score', 'rating_name', 'effect', 'stars', 'forks', 'language'
]].rename(columns={
    'name': 'Skill名称',
    'scenario_category': '场景类别',
    'recommend_score': '推荐得分',
    'rating_name': '评级',
    'effect': '具体作用',
    'stars': 'Stars',
    'forks': 'Forks',
    'language': '语言',
})
display(top_table)

official_rows = []
for company in data.get('official_companies', []):
    examples = company.get('examples', [])[:3]
    row = {'公司/机构': company.get('company', '—'), 'Skill数量': company.get('count', 0)}
    for index in range(3):
        example = examples[index] if index < len(examples) else {}
        name = example.get('name', '—')
        effect = example.get('effect', '—')
        row[f'代表Skill{index + 1}'] = f'{name} - {effect}' if name != '—' else '—'
    official_rows.append(row)
official_collection = pd.DataFrame(official_rows).sort_values('Skill数量', ascending=False)
display(official_collection)


In [ ]:
community_rows = []
community_summary_rows = []
for category in data.get('community_categories', []):
    category_name = category.get('category', '—')
    all_recommended = category.get('recommended', [])
    high_score_items = [item for item in all_recommended if (item.get('recommend_score') or 0) >= 70]
    selected_items = high_score_items[:5]
    recommendation_type = '正式推荐'
    if not selected_items and all_recommended:
        selected_items = all_recommended[:1]
        recommendation_type = '候选推荐'
    if not selected_items:
        continue
    community_summary_rows.append({
        '场景类别': category_name,
        '原始Skill数量': category.get('count', 0),
        '推荐数量': len(selected_items),
        '最高得分': max(item.get('recommend_score') or 0 for item in selected_items),
        '推荐类型': recommendation_type,
        '代表Skill示例': '；'.join(item.get('name', '—') for item in selected_items[:3]),
    })
    for rank, item in enumerate(selected_items, start=1):
        metrics = item.get('github_metrics') or {}
        community_rows.append({
            '场景类别': item.get('scenario_category') or category_name,
            '类别内排名': rank,
            'Skill名称': item.get('name', '—'),
            '链接': item.get('link', ''),
            '推荐得分': item.get('recommend_score', 0),
            '评级': f"{item.get('rating_stars', '')} {item.get('rating_name', '')}".strip(),
            '具体作用': item.get('effect', '—'),
            'Stars': metrics.get('stars') or 0,
            'Forks': metrics.get('forks') or 0,
            '语言': metrics.get('language') or '未知',
        })

community_summary_table = pd.DataFrame(community_summary_rows).sort_values(['推荐类型', '最高得分'], ascending=[True, False])
community_collection_table = pd.DataFrame(community_rows)
display(community_summary_table)
display(community_collection_table)


## 十二、结论与后续改进方向

1. 本项目将分散的 AI Agent Skills 数据整理为可分析、可推荐的结构化数据。
2. 通过领域划分，可以把社区 Skill 组织成安全、科研、开发、自动化、RAG、内容创作、营销增长等多个推荐集合。
3. 通过综合评分，可以在每个领域中筛选出更值得优先关注的 Skill，避免只依赖人工主观判断。
4. 所有图表和结果表格均在 notebook 中直接生成，不依赖外部图表脚本或输出目录。
5. 后续可以继续优化推荐模型，例如增加 Issue 活跃度、贡献者数量、下载量、README 语义质量等指标。
